# 006 Custom Workflow

这是 LangChain Multi-agent 学习线的第六份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/multi-agent/custom-workflow

学习目标：

1. 理解 custom workflow 是显式控制流，不是自由 agent 循环
2. 学会用 LangGraph `StateGraph` 定义稳定流程
3. 学会把 research、implementation、verification、synthesis 分成节点
4. 学会用条件边决定是否进入工程工作流
5. 对比本仓库 Harness Query Loop 和 LangGraph workflow

这一讲使用确定性 Python 节点，不消耗真实模型额度。

## 1. Custom workflow 解决什么问题

前面几讲分别学习了：

- subagent：委派局部任务
- handoff：交接后续控制权
- skill：按需加载能力包
- router：入口分类和分发

custom workflow 解决的是更明确的问题：

```text
当任务流程本身比较稳定时，
不要完全交给 agent 自由决定下一步，
而是由系统显式定义步骤、分支和停止条件。
```

这和 Java 里的工作流/流程编排很像：

```text
Controller -> Service A -> Service B -> Validator -> Response Builder
```

只是在 LangGraph 里，每一步可以是普通函数，也可以是 agent。

## 2. 什么时候应该用 Custom workflow

适合用 custom workflow 的场景：

1. 步骤顺序稳定
2. 有明确完成标准
3. 需要独立验证
4. 出错后需要知道是哪一段出错
5. 需要把 agent 能力限制在固定流程里

不适合的场景：

1. 普通闲聊
2. 一次工具调用就能解决的问题
3. 用户意图完全不稳定，流程每次都不一样

一句话：

```text
流程稳定时，用 workflow。
步骤未知时，用 agent loop。
```

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class EngineeringWorkflowState(TypedDict):
    user_message: str
    route: str
    research: str
    implementation: str
    verification: str
    attempts: int
    answer: str


def print_state(result: EngineeringWorkflowState) -> None:
    for key in ["route", "research", "implementation", "verification", "attempts", "answer"]:
        print(key + ":", result.get(key, ""))

## 3. 定义 workflow 节点

这个示例把工程任务拆成四个稳定节点：

```text
research -> implementation -> verification
                         ^        |
                         |        v
                         retry  synthesis
```

另外加一个 `planner_node` 做入口判断：

- 如果问题像工程任务，进入 workflow
- 如果只是普通问题，直接回答

注意：这里的节点都是普通 Python 函数。真实系统里可以把其中某些节点换成 agent。

In [ ]:
def planner_node(state: EngineeringWorkflowState) -> dict:
    message = state["user_message"].lower()
    engineering_keywords = ["bug", "代码", "改造", "测试", "实现", "harness", "workflow"]

    if any(keyword in message for keyword in engineering_keywords):
        return {"route": "engineering"}

    return {
        "route": "direct_answer",
        "answer": "这是普通问题，不需要进入工程工作流。",
    }


def research_node(state: EngineeringWorkflowState) -> dict:
    return {
        "research": "已确认相关区域是 app/agents/harness.py 和 notebooks/langchain-multi-agent。"
    }


def implementation_node(state: EngineeringWorkflowState) -> dict:
    attempts = state.get("attempts", 0) + 1
    if attempts == 1:
        implementation = "第一次实现：完成主体逻辑，但还缺少验证脚本。"
    else:
        implementation = "第二次实现：补充验证脚本和边界说明。"

    return {
        "attempts": attempts,
        "implementation": implementation,
    }


def verification_node(state: EngineeringWorkflowState) -> dict:
    ok = state.get("attempts", 0) >= 2
    return {
        "verification": "PASS: 改造有明确 research 依据，并完成最小验证。" if ok else "FAIL: 缺少验证脚本，需要回到 implementation。"
    }


def synthesis_node(state: EngineeringWorkflowState) -> dict:
    return {
        "answer": "\n".join(
            [
                "工程工作流已完成：",
                "1. " + state.get("research", ""),
                "2. " + state.get("implementation", ""),
                "3. " + state.get("verification", ""),
            ]
        )
    }

## 4. 定义条件边

workflow 的关键不是节点，而是边。

这里 `choose_after_planner` 决定：

- `engineering`：进入 research 节点
- `direct_answer`：直接结束

这就是显式控制流。

In [ ]:
def choose_after_planner(state: EngineeringWorkflowState) -> Literal["engineering", "direct_answer"]:
    if state["route"] == "engineering":
        return "engineering"
    return "direct_answer"


def choose_after_verification(state: EngineeringWorkflowState) -> Literal["retry", "synthesis"]:
    if state["verification"].startswith("PASS"):
        return "synthesis"
    return "retry"

## 5. 组装 LangGraph workflow

下面的图结构是：

```text
START
  -> planner
    -> direct_answer -> END
    -> research -> implementation -> verification
                                      -> retry implementation
                                      -> synthesis -> END
```

In [ ]:
workflow_builder = StateGraph(EngineeringWorkflowState)

workflow_builder.add_node("planner", planner_node)
workflow_builder.add_node("research", research_node)
workflow_builder.add_node("implementation", implementation_node)
workflow_builder.add_node("verification", verification_node)
workflow_builder.add_node("synthesis", synthesis_node)

workflow_builder.add_edge(START, "planner")
workflow_builder.add_conditional_edges(
    "planner",
    choose_after_planner,
    {
        "engineering": "research",
        "direct_answer": END,
    },
)
workflow_builder.add_edge("research", "implementation")
workflow_builder.add_edge("implementation", "verification")
workflow_builder.add_conditional_edges(
    "verification",
    choose_after_verification,
    {
        "retry": "implementation",
        "synthesis": "synthesis",
    },
)
workflow_builder.add_edge("synthesis", END)

engineering_workflow = workflow_builder.compile()
engineering_workflow

## 6. 跑通工程任务

这个输入命中工程关键词，所以会经过完整流程：

```text
planner -> research -> implementation -> verification -> synthesis
```

In [ ]:
engineering_result = engineering_workflow.invoke(
    {
        "user_message": "请帮我改造 Harness workflow，并补充测试验证",
        "route": "",
        "research": "",
        "implementation": "",
        "verification": "",
        "attempts": 0,
        "answer": "",
    }
)

print_state(engineering_result)

## 7. 跑通普通问题

这个输入不命中工程关键词，所以不会进入 research / implementation / verification。

In [ ]:
direct_result = engineering_workflow.invoke(
    {
        "user_message": "Python 的 lower 是什么意思？",
        "route": "",
        "research": "",
        "implementation": "",
        "verification": "",
        "attempts": 0,
        "answer": "",
    }
)

print_state(direct_result)

## 8. Custom workflow 和 Router 的区别

| 模式 | 主要职责 | 控制粒度 |
| --- | --- | --- |
| Router | 决定进入哪条路径 | 入口级 |
| Custom workflow | 定义路径里的每一步怎么走 | 流程级 |

在本讲示例里：

```text
planner_node 像 router，决定是否进入工程路径。
StateGraph 是 custom workflow，定义进入后必须走哪些步骤。
```

所以 Router 可以是 Custom workflow 的一个节点，但 Router 本身不等于 Custom workflow。

## 9. 和本仓库 Harness Query Loop 的关系

本仓库 Harness Query Loop 更像一个受控 agent loop：

```text
planner -> action -> tool/delegate/answer -> ledger -> continue or stop
```

LangGraph Custom workflow 更像显式流程图：

```text
node A -> node B -> conditional edge -> node C -> END
```

两者可以结合：

- Harness Query Loop 适合开放式任务处理
- Custom workflow 适合稳定业务流程
- Harness 的 research / implementation / verification 分区，可以落成 LangGraph 节点
- approval、verification、synthesis 可以作为固定流程门禁

## 10. Custom workflow 的设计要点

设计 custom workflow 时要明确：

1. state 里有哪些字段？
2. 每个节点只负责什么？
3. 哪些节点可以是普通函数？
4. 哪些节点需要 agent？
5. 哪些边是固定顺序？
6. 哪些边是条件分支？
7. 失败时是重试、人工审批、还是终止？
8. 最终由谁 synthesis？

不要为了显得“智能”而放弃稳定流程。

## 11. 本讲练习

请判断下面场景更适合 agent loop 还是 custom workflow：

1. 用户随便问一个开放问题。
2. 每次代码改造都必须先 research，再 implementation，再 verification，再 synthesis。
3. 报销单必须经过 OCR、金额校验、权限校验、审批、入账。
4. 用户临时问“这个函数是什么意思”。

参考答案：

1. agent loop
2. custom workflow
3. custom workflow
4. agent loop 或直接回答

## 12. 本讲小结

这一讲的核心：

```text
Custom workflow 是显式控制流。
```

你现在应该能判断：

- 什么时候不要让 agent 自由决定下一步
- `StateGraph` 如何定义状态、节点和边
- 条件边如何控制流程分支
- research / implementation / verification / synthesis 如何落成稳定节点
- Harness Query Loop 和 LangGraph workflow 如何互补

到这里，LangChain Multi-agent 的六个基础模式已经走完。